# Stage 3 — Enrichment + embeddings in one Ray Data graph

Run the Stage 2 LoRA-adapted Qwen2.5-VL-3B student to emit catalog JSON, *and* SigLIP-2 (so400m, 1152-dim) image and text embeddings — all in a single streaming Ray Data pipeline. No intermediate disk writes between the VLM and the embedding stages.

This notebook runs the same code paths as `scripts/run_enrich_and_embed.py` at a small N. If the Stage 2 adapter dir is present, the pipeline routes through the fine-tuned weights; otherwise it falls back to the base 3B model. For the end-to-end story see the template's [`README.ipynb`](../README.ipynb).

In [ ]:
import os, sys, json

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

# MUST be set before importing the script — the LoRA detection runs at
# module-import time. If the dir doesn't exist the script falls back to base 3B.
os.environ.setdefault(
    "QWEN_LORA_ADAPTER_DIR",
    "/mnt/cluster_storage/vlm-distillation-catalog-enrichment/qwen25vl_3b_enrichment_lora_smoke",
)

import ray
if ray.is_initialized():
    ray.shutdown()

ray.init(
    runtime_env={
        "working_dir": os.path.abspath(os.path.join(os.getcwd(), "..")),
    },
)
print("Cluster resources:", json.dumps(ray.cluster_resources(), indent=2))

In [ ]:
from scripts import run_enrich_and_embed as enc

# Smoke overrides — same code paths as the production job, just smaller. We
# rebind module attributes so the script's helpers (build_catalog,
# build_vlm_processor) pick them up at call time via the module globals.
N_SMOKE = 50
enc.N_ROWS = N_SMOKE
enc.VLM_CONCURRENCY = 1
enc.EMB_CONCURRENCY = 1
enc.PROCESS_CONCURRENCY = 4
enc.FETCH_CONCURRENCY = 8
enc.CACHE_PATH = f"{enc.BASE_DIR}/catalog_{enc.CATEGORY}_{N_SMOKE}.parquet"
enc.CHECKPOINT_PATH = f"{enc.BASE_DIR}/enc_vlm_emb_enrich_smoke_{N_SMOKE}_checkpoint"
enc.OUTPUT_PATH = f"{enc.BASE_DIR}/enc_vlm_enriched_with_embeddings_smoke_{N_SMOKE}.parquet"

vlm_mode = (
    f"LoRA: {enc.LORA_ADAPTER_NAME}  (loaded from {enc.LORA_REMOTE_PREFIX})"
    if enc.LORA_ADAPTER_NAME
    else f"base 3B ({enc.VLM_MODEL_SOURCE})"
)
print(f"smoke config: N={N_SMOKE}  VLM_CONCURRENCY={enc.VLM_CONCURRENCY}  EMB_CONCURRENCY={enc.EMB_CONCURRENCY}")
print(f"VLM mode:     {vlm_mode}")
print(f"output:       {enc.OUTPUT_PATH}")

In [ ]:
import io as _io
import requests as _r
from PIL import Image as _Image

# build_catalog() caches at CACHE_PATH; first run does the HF read + URL HEAD
# checks, repeat runs reuse the parquet. Returns a Ray Dataset.
ds = enc.build_catalog()
print(f"\n[catalog] {ds.count()} rows after filtering")

row = ds.take(1)[0]
print("\n[input row]")
for k, v in row.items():
    print(f"  {k:14} = {repr(v)[:100]}")

resp = _r.get(row["image_url"], timeout=5, headers={"User-Agent": "vlm-distillation-catalog-enrichment/1.0"})
img = _Image.open(_io.BytesIO(resp.content))
img.thumbnail((256, 256))
img

In [ ]:
from ray.data.checkpoint import CheckpointConfig

# CPU fetch + decode + resize
ds = ds.map_batches(
    enc.fetch_and_resize,
    batch_size=16,
    concurrency=enc.FETCH_CONCURRENCY,
    batch_format="numpy",
)

# VLM enrichment via ray.data.llm (vLLM under the hood; LoRA-routed if attached)
ds = enc.build_vlm_processor()(ds)

# SigLIP CPU process — image processor + text tokenizer over title + VLM description
ds = ds.map(
    enc.ProcessSigLIP,
    fn_constructor_kwargs={"model_id": enc.EMB_MODEL_SOURCE},
    num_cpus=1,
    concurrency=enc.PROCESS_CONCURRENCY,
)

# SigLIP GPU embed — pure forward pass on pre-tensorized inputs
ds = ds.map_batches(
    enc.EmbedSigLIP,
    fn_constructor_kwargs={"model_id": enc.EMB_MODEL_SOURCE},
    batch_size=enc.EMB_BATCH_SIZE,
    num_gpus=1,
    concurrency=enc.EMB_CONCURRENCY,
    batch_format="numpy",
)

# Set CheckpointConfig AFTER build_catalog so the catalog's parquet write
# doesn't poison the inference checkpoint with already-seen IDs (would skip
# ~99% of rows on the next run).
ctx = ray.data.DataContext.get_current()
ctx.checkpoint_config = CheckpointConfig(
    id_column="id",
    checkpoint_path=enc.CHECKPOINT_PATH,
    delete_checkpoint_on_success=False,
)

ds.write_parquet(enc.OUTPUT_PATH)
print(f"[done] wrote {enc.OUTPUT_PATH}")

In [ ]:
sample = ray.data.read_parquet(enc.OUTPUT_PATH).take(2)
print(f"[output] read back {len(sample)} sample rows from {enc.OUTPUT_PATH}\n")

for r in sample:
    print(json.dumps({
        "title":               r["title"],
        "raw_output":          r["raw_output"],
        "image_embedding[:6]": [round(float(x), 3) for x in r["image_embedding"][:6]],
        "text_embedding[:6]":  [round(float(x), 3) for x in r["text_embedding"][:6]],
    }, indent=2))
    print()